<div style="font-size: 1em; display: flex; align-items: center; gap: 8px; padding: 8px 16px; background: #F8F9FA; border-bottom: 2px solid #E0E0E0; margin: 0; line-height: 1">
    <img src="https://cdn.simpleicons.org/databricks/FF3621" width="24" height="24"/>
    <div style="color: #666">
        <span style="font-weight: bold; color: #333">Data Interoperability with Unity Catalog</span>
    </div>
</div>

<p style="font-size: 1em; text-align: center; line-height: 0; padding-top: 9px; margin: 4px 0">
<img
src="https://databricks.com/wp-content/uploads/2018/03/db-academy-rgb-1200px.png"
alt="Databricks Learning"
>
</div>

# Lab Environment Overview

This notebook describes the compute requirements, classroom setup options, and data environment for the demos and labs in the **Data Interoperability with Unity Catalog** course. Review it before starting any notebooks.

<div style="font-size: 1em; border-left: 4px solid #1976d2; background: #e3f2fd; padding: 16px 20px; border-radius: 4px; margin: 16px 0;">
    <div style="display: flex; align-items: flex-start; gap: 12px;">
        <div>
            <strong style="color: #0d47a1; font-size: 1.1em;">Use Compute Types as Directed by Each Notebook</strong>
            <p style="margin: 8px 0 0 0; color: #333;">This course uses <b>Serverless SQL Warehouse</b> for most demos and labs, and <b>Serverless notebook compute</b> for the cells that use Python (PyIceberg). Follow the compute guidance in each notebook.</p>
        </div>
    </div>
</div>

<div style="font-size: 1em; border-left: 4px solid #ff9800; background: #fff3e0; padding: 16px 20px; border-radius: 4px; margin: 16px 0;">
    <div style="display: flex; align-items: flex-start; gap: 12px;">
        <div>
            <strong style="color: #e65100; font-size: 1.1em;">Prerequisites - Run These First</strong>
            <p style="margin: 8px 0 0 0; color: #333;"><ol style="margin: 8px 0 0 0;">
<li><b>0 - Required Setup</b> - run by every student at the start of the lab session. Creates the catalog, the <code>data_interoperability_tpcds</code> schema, and the dimension views.</li>
<li><b>1 - Instructor Demo Setup</b> - run by the <b>instructor only</b>, before class starts. Materializes the two large reference tables (<code>store_sales_unclustered</code> and <code>store_sales_clustered</code>, ~2.8 billion rows each) used by the Module 2 performance demo. Allow up to 10 minutes.</li>
</ol></p>
        </div>
    </div>
</div>

## Classroom Setup

| Option | Workspace Type | Catalog Setup | Catalog Name Format |
|--------|----------------|---------------|---------------------|
| **Option 1** | Databricks Academy Provided Workspace (Vocareum) | Already created for you | `labuser12345` (matches your Vocareum username) |
| **Option 2** | Other Workspaces or Databricks Free Edition | Setup will create a Unity Catalog catalog for you (**Create catalog permission required**) | `labuser_username` (derived from your Databricks username) |

<div style="font-size: 1em; border-left: 4px solid #f44336; background: #ffebee; padding: 16px 20px; border-radius: 4px; margin: 16px 0;">
    <div style="display: flex; align-items: flex-start; gap: 12px;">
        <div>
            <strong style="color: #c62828; font-size: 1.1em;">Do Not Run in Production Environments</strong>
            <p style="margin: 8px 0 0 0; color: #333;"><ul><li>Only run these notebooks in <b>development or sandbox workspaces</b>.</li><li>The setup script creates schemas and tables in your personal catalog.</li></ul></p>
        </div>
    </div>
</div>

## Environment Overview

Each demo and lab notebook includes a setup cell near the top that initializes your environment:

```
%run ../Includes/Classroom-Setup-X
```

This runs the classroom setup script which: detects your workspace type (Vocareum or other), configures your Unity Catalog, sets your default catalog and schema for the session, and reports setup status.

## Schemas and Datasets

The course follows a single steel-thread dataset built on TPC-DS at scale factor 1000 (~2.8 billion-row `store_sales`). The `0 - Required Setup` notebook seeds the schema and dimension views; the instructor pre-builds the two large performance-demo tables; the `2.2 Demo` creates the Iceberg and UniForm variants that the rest of the course consumes.

All artefacts live in a single schema in your personal catalog: `data_interoperability_tpcds`.

| Object | Created By | Used In | Description |
|--------|-----------|---------|-------------|
| `date_dim`, `item`, `customer`, `store` | `0 - Required Setup` | Modules 3-5 | Views over `samples.tpcds_sf1000.*` dimensions. |
| `store_sales_unclustered` | `1 - Instructor Demo Setup` | Modules 2-5 | Shuffled CTAS of `samples.tpcds_sf1000.store_sales` (~2.8B rows). Baseline for the Module 2 perf demo and source for module-specific CTASes. |
| `store_sales_clustered` | `1 - Instructor Demo Setup` | Module 2 | Same data, clustered by `(ss_sold_date_sk, ss_item_sk)`. |
| `store_sales_iceberg` | `2.2 Demo` | Modules 3, 4, 5 | Managed Iceberg with same clustering. Read by Snowflake (3.2) and EMR (4.2); joined federated in 5.5. |
| `store_sales_delta_uniform` | `2.2 Demo` | Module 4 | Delta + UniForm variant; loadable from external Iceberg engines. |
| `store_sales_delta_plain` | `2.2 Demo` | Module 2 | 1M-row plain Delta sample used as the negative case in the PyIceberg cross-format test. |
| `item_demo` | `Classroom-Setup-3-Iceberg` | Module 3 | Small Delta CTAS from `samples.tpcds_sf1000.item` (~300K rows) used as the V3 read-enable subject in 3.3 Lab. |
| `item_iceberg`, `item_uniform` | `3.3 Lab` | Module 3 | Managed Iceberg and Delta + UniForm copies of the same source created by lab tasks for the PyIceberg cross-format check. |

## Compute Requirements

| Module | Compute Type | Minimum Version |
|--------|-------------|----------------|
| Module 2 (Demo: 2.2) | Serverless SQL Warehouse and Serverless notebook (PyIceberg cell uses Python) | Default |
| Module 3 (Demos and Labs) | Serverless SQL Warehouse; Lab 3.4 uses Serverless notebook for PyIceberg | Default |
| Module 4 (EMR Demo) | Serverless SQL Warehouse for the Databricks-side cells; EMR cluster for the external-side cells | EMR 7.x with Iceberg 1.6+ |
| Module 5 (Federation) | SQL Warehouse (Pro or Serverless) | 2024.x+ |

### Using an Existing Catalog

If you prefer to use an existing catalog, the `Classroom-Setup-Common` notebook will detect your existing catalog or create one. To force a specific catalog, modify the `catalog_forced` variable in the setup.

&copy; 2026 Databricks, Inc. All rights reserved. Apache, Apache Spark, Spark, the Spark Logo, Apache Iceberg, Iceberg, and the Apache Iceberg logo are trademarks of the <a href="https://www.apache.org/" target="_blank">Apache Software Foundation</a>.<br/><br/><a href="https://databricks.com/privacy-policy" target="_blank">Privacy Policy</a> | <a href="https://databricks.com/terms-of-use" target="_blank">Terms of Use</a> | <a href="https://help.databricks.com/" target="_blank">Support</a>